In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras import layers, models
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, project_root)
import importlib
import models.pretrained_resnet as resnet
importlib.reload(resnet)
import utilities.util_metrics as metrics
importlib.reload(metrics)
import utilities.util_dataloader as dataloader
importlib.reload(dataloader)

In [ ]:
#data
from importlib import util


batch_size = 16
img_size = (224, 224)
directory = 'C:\\Users\\frank\\OneDrive\\Documents\\vsCodeProjects\\cytology\\data'
seed = 100

train_ds = tf.keras.utils.image_dataset_from_directory(
    directory,
    validation_split=0.4,  
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)


temp_ds = tf.keras.utils.image_dataset_from_directory(
    directory,
    validation_split=0.4,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
temp_ds = temp_ds.cache().prefetch(buffer_size=AUTOTUNE)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
])


X_train, y_train = util.dataloaderNumpy(train_ds)
X_val, y_val = util.dataloaderNumpy(temp_ds)

print(X_train.shape, y_train.shape)

In [ ]:

#Build
X_train = preprocess_input(X_train)
X_val = preprocess_input(X_val)
model = resnet.build_resnet(input_shape=X_train.shape[1:], num_classes=len(np.unique(y_train)))

In [ ]:
#Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
#optimize
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_control_cnn.h5",
    monitor='val_accuracy',
    save_best_only=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=16,
    callbacks=[early_stop, checkpoint]
)

In [ ]:
#prediction
y_pred_probs = model.predict(X_val)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = y_val

In [ ]:
#results 
import json


acc = metrics.compute_accuracy(y_true, y_pred)
f1 = metrics.compute_f1(y_true, y_pred)
auc_score = metrics.compute_auc(y_true, y_pred_probs, num_classes=3)

print("Accuracy:", acc)
print("F1 Score:", f1)
print("AUC:", auc_score)

class_names = ["NILM", "LSIL", "HSIL"]

metrics.plot_confusion_matrix(y_true, y_pred, class_names)
metrics.plot_roc_multiclass(y_true, y_pred_probs, 3, class_names)

results = {
    "model_name": "resnet50", 
    "accuracy": acc,
    "f1_score": f1,
    "auc": auc_score
}

print(json.dumps(results))